## Project Dataset Analysis
- Part 0: Load & Rename
- Part 1: Patron Retention 
- Part 2: SBK's Best & Worst Month
- Part 3: Parlays vs. Straights

## Part 0: Load & Rename

In [1]:
import pandas as pd
import duckdb
pd.options.display.float_format = '{:,.2f}'.format 

In [2]:
df = pd.read_csv ("ProjectDataset.csv")

In [3]:
# rename columns to what I'm used to working with at theScore
rename_map = {
    "playerid": "player_id",
    "wagerid": "wager_id",
    "sportname": "sport",
    "result": "bet_result",
    "legresult": "leg_result",
    "decimalodds": "decimal_odds",
    "net_stake": "handle",
    "decimal_odds": "odds"
}

df = df.rename(columns=rename_map)
df.columns

Index(['state', 'player_id', 'wager_id', 'event_start', 'placed_date',
       'settled_date', 'sport', 'bet_type', 'bet_result', 'handle', 'ggr',
       'leg_result', 'decimal_odds'],
      dtype='object')

In [4]:
# while the datset includes a few days at the end of March 2021, I prefer to work with a clean 12 month date range.
df = df[
    (df["placed_date"] >= "2021-04-01") &
    (df["placed_date"] <= "2022-03-31")
].copy()

In [5]:
duckdb.register('bets', df)

## Part 1: Patron Retention

**1a: Given the year dataset provided (April 2021 -> March 2022), how many months were players active?**

In [6]:
duckdb.sql ("""
WITH sports_encoded AS (
        SELECT *
        , CASE WHEN LOWER (sport) == 'nhl' THEN 1 ELSE 0 END AS is_nhl_bet
        , CASE WHEN LOWER (sport) == 'nba' THEN 1 ELSE 0 END AS is_nba_bet
        , CASE WHEN LOWER (sport) == 'mlb' THEN 1 ELSE 0 END AS is_mlb_bet
        , CASE WHEN LOWER (sport) == 'champions league' THEN 1 ELSE 0 END AS is_champions_league_bet
        , CASE WHEN LOWER (sport) == 'nfl' THEN 1 ELSE 0 END AS is_nfl_bet
        , CASE WHEN LOWER (sport) == 'college football' THEN 1 ELSE 0 END AS is_college_football_bet
        , CASE WHEN LOWER (sport) == 'college basketball' THEN 1 ELSE 0 END AS is_college_basketball_bet  
        FROM bets
        )
, wager_level AS (
        SELECT player_id
        , wager_id
        , bet_type
        , handle
        , ggr
        , MIN(CAST (placed_date AS TIMESTAMP)) AS placed_date
        , MAX(CAST (settled_date AS TIMESTAMP)) AS settled_date
        , MAX(CAST (event_start AS TIMESTAMP)) AS event_start
        , MAX(is_nhl_bet) AS is_nhl_bet
        , MAX(is_nba_bet) AS is_nba_bet
        , MAX(is_mlb_bet) AS is_mlb_bet
        , MAX(is_champions_league_bet) AS is_champions_league_bet
        , MAX(is_nfl_bet) AS is_nfl_bet
        , MAX(is_college_football_bet) AS is_college_football_bet
        , MAX(is_college_basketball_bet) AS is_college_basketball_bet
        FROM sports_encoded
        GROUP BY 1, 2, 3, 4, 5          
        )
, player_months AS (
        SELECT player_id
        , COUNT(DISTINCT date_trunc('month', placed_date)) AS active_months
        FROM wager_level
        GROUP BY 1
        )
, player_buckets AS (
        SELECT player_id
        , active_months
        , CASE
                WHEN active_months == 1 THEN '1 month'
                WHEN active_months == 2 THEN '2 month'
                WHEN active_months == 3 THEN '3 month'
                WHEN active_months == 4 THEN '4 month'
                WHEN active_months == 5 THEN '5 month'
                ELSE '6+ months'
        END AS retention_bucket
        FROM player_months
            )
SELECT
retention_bucket
, COUNT(*) AS num_players
, ROUND(COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct_players 
FROM player_buckets
GROUP BY 1 
ORDER BY 1
            
            """).df()

,retention_bucket,num_players,pct_players
0,1 month,11819,0.33
1,2 month,5776,0.16
2,3 month,3751,0.10
3,4 month,2702,0.07
4,5 month,2535,0.07
5,6+ months,9701,0.27


**1b: What differences do we see in the betting tendencies of 1-3 month bettors vs. loyal bettors of 6 months+?**

In [7]:
duckdb.sql ("""
WITH sports_encoded AS (
        SELECT *
        , CASE WHEN LOWER (sport) == 'nhl' THEN 1 ELSE 0 END AS is_nhl_bet
        , CASE WHEN LOWER (sport) == 'nba' THEN 1 ELSE 0 END AS is_nba_bet
        , CASE WHEN LOWER (sport) == 'mlb' THEN 1 ELSE 0 END AS is_mlb_bet
        , CASE WHEN LOWER (sport) == 'champions league' THEN 1 ELSE 0 END AS is_champions_league_bet
        , CASE WHEN LOWER (sport) == 'nfl' THEN 1 ELSE 0 END AS is_nfl_bet
        , CASE WHEN LOWER (sport) == 'college football' THEN 1 ELSE 0 END AS is_college_football_bet
        , CASE WHEN LOWER (sport) == 'college basketball' THEN 1 ELSE 0 END AS is_college_basketball_bet  
        FROM bets
        )
, wager_level AS (
        SELECT player_id
        , wager_id
        , bet_type
        , handle
        , ggr
        , MIN(CAST (placed_date AS TIMESTAMP)) AS placed_date
        , MAX(CAST (settled_date AS TIMESTAMP)) AS settled_date
        , MAX(CAST (event_start AS TIMESTAMP)) AS event_start
        , MAX(is_nhl_bet) AS is_nhl_bet
        , MAX(is_nba_bet) AS is_nba_bet
        , MAX(is_mlb_bet) AS is_mlb_bet
        , MAX(is_champions_league_bet) AS is_champions_league_bet
        , MAX(is_nfl_bet) AS is_nfl_bet
        , MAX(is_college_football_bet) AS is_college_football_bet
        , MAX(is_college_basketball_bet) AS is_college_basketball_bet
        FROM sports_encoded
        GROUP BY 1, 2, 3, 4, 5          
        )
, player_months AS (
        SELECT player_id
        , COUNT (DISTINCT date_trunc ('month', placed_date)) AS active_months
        FROM wager_level
        GROUP BY 1
        )
, player_buckets AS (
        SELECT player_id
        , active_months
        , CASE
                WHEN active_months == 1 THEN '1-3 months'
                WHEN active_months == 2 THEN '1-3 months'
                WHEN active_months == 3 THEN '1-3 months'
                WHEN active_months == 4 THEN '4 month'
                WHEN active_months == 5 THEN '5 month'
                ELSE '6+ months'
        END AS retention_bucket
        FROM player_months
            )
, short_long AS (
        SELECT player_id
        , active_months
        , retention_bucket
        FROM player_buckets
        WHERE retention_bucket IN ('1-3 months', '6+ months')
        )
, short_long_wagers AS (
        SELECT sl.player_id
        , sl.active_months
        , sl.retention_bucket
        , w.wager_id
        , w.placed_date
        , w.bet_type
        , w.handle
        , w.ggr
        , is_nhl_bet
        , is_nba_bet
        , is_mlb_bet
        , is_champions_league_bet
        , is_nfl_bet
        , is_college_football_bet
        , is_college_basketball_bet
        FROM short_long sl
        LEFT JOIN wager_level w ON sl.player_id = w.player_id
        )
, monthly_betting AS (
        SELECT player_id
        , active_months
        , retention_bucket
        , date_trunc('month', placed_date) as month
        , COUNT(*) as monthly_bets
        FROM short_long_wagers
        GROUP BY 1, 2, 3, 4
        )        
, per_player_avg AS (
        SELECT player_id
        , retention_bucket
        , AVG(monthly_bets) as avg_monthly_bets_per_player
        FROM monthly_betting
        GROUP BY 1, 2
        )
, bucket_avg AS (
        SELECT retention_bucket
        , AVG(avg_monthly_bets_per_player) AS avg_monthly_bets_in_bucket
        FROM per_player_avg
        GROUP BY 1
        )
            
SELECT slw.retention_bucket
,  ba.avg_monthly_bets_in_bucket AS avg_monthly_bets
, AVG(slw.handle) AS avg_bet_size
FROM short_long_wagers slw
LEFT JOIN bucket_avg ba ON slw.retention_bucket = ba.retention_bucket
GROUP BY 1, 2
ORDER BY 1

            """).df()

,retention_bucket,avg_monthly_bets,avg_bet_size
0,1-3 months,6.93,40.34
1,6+ months,21.97,30.76


**1c: What differences do we see in sports bet on?**

In [8]:
duckdb.sql ("""
WITH sports_encoded AS (
        SELECT *
        , CASE WHEN LOWER (sport) == 'nhl' THEN 1 ELSE 0 END AS is_nhl_bet
        , CASE WHEN LOWER (sport) == 'nba' THEN 1 ELSE 0 END AS is_nba_bet
        , CASE WHEN LOWER (sport) == 'mlb' THEN 1 ELSE 0 END AS is_mlb_bet
        , CASE WHEN LOWER (sport) == 'champions league' THEN 1 ELSE 0 END AS is_champions_league_bet
        , CASE WHEN LOWER (sport) == 'nfl' THEN 1 ELSE 0 END AS is_nfl_bet
        , CASE WHEN LOWER (sport) == 'college football' THEN 1 ELSE 0 END AS is_college_football_bet
        , CASE WHEN LOWER (sport) == 'college basketball' THEN 1 ELSE 0 END AS is_college_basketball_bet  
        FROM bets
        )
, wager_level AS (
        SELECT player_id
        , wager_id
        , bet_type
        , handle
        , ggr
        , MIN(CAST (placed_date AS TIMESTAMP)) AS placed_date
        , MAX(CAST (settled_date AS TIMESTAMP)) AS settled_date
        , MAX(CAST (event_start AS TIMESTAMP)) AS event_start
        , MAX(is_nhl_bet) AS is_nhl_bet
        , MAX(is_nba_bet) AS is_nba_bet
        , MAX(is_mlb_bet) AS is_mlb_bet
        , MAX(is_champions_league_bet) AS is_champions_league_bet
        , MAX(is_nfl_bet) AS is_nfl_bet
        , MAX(is_college_football_bet) AS is_college_football_bet
        , MAX(is_college_basketball_bet) AS is_college_basketball_bet
        FROM sports_encoded
        GROUP BY 1, 2, 3, 4, 5          
        )
, player_months AS (
        SELECT player_id
        , COUNT(DISTINCT date_trunc('month', placed_date)) AS active_months
        FROM wager_level
        GROUP BY 1
        )
, player_buckets AS (
        SELECT player_id
        , active_months
        , CASE
                WHEN active_months == 1 THEN '1-3 months'
                WHEN active_months == 2 THEN '1-3 months'
                WHEN active_months == 3 THEN '1-3 months'
                WHEN active_months == 4 THEN '4 month'
                WHEN active_months == 5 THEN '5 month'
                ELSE '6+ months'
        END AS retention_bucket
        FROM player_months
            )
, short_long AS (
        SELECT player_id
        , active_months
        , retention_bucket
        FROM player_buckets
        WHERE retention_bucket IN ('1-3 months', '6+ months')
        )
, short_long_wagers AS (
        SELECT sl.player_id
        , sl.active_months
        , sl.retention_bucket
        , w.wager_id
        , w.placed_date
        , w.bet_type
        , w.handle
        , w.ggr
        , is_nhl_bet
        , is_nba_bet
        , is_mlb_bet
        , is_champions_league_bet
        , is_nfl_bet
        , is_college_football_bet
        , is_college_basketball_bet
        FROM short_long sl
        LEFT JOIN wager_level w ON sl.player_id = w.player_id
        )
            
SELECT retention_bucket
    , ROUND(SUM(is_nhl_bet) / COUNT(*), 4) * 100 AS pct_nhl_bets
    , ROUND(SUM(is_nba_bet) / COUNT(*), 4) * 100 AS pct_nba_bets
    , ROUND(SUM(is_mlb_bet) / COUNT(*), 4) * 100 AS pct_mlb_bets
    , ROUND(SUM(is_champions_league_bet) / COUNT(*), 4) * 100 AS pct_champions_league_bets
    , ROUND(SUM(is_nfl_bet) / COUNT(*), 4) * 100 AS pct_nfl_bets
    , ROUND(SUM(is_college_football_bet) / COUNT(*), 4) * 100 AS pct_college_football_bets
    , ROUND(SUM(is_college_basketball_bet) / COUNT(*), 4) * 100 AS pct_college_basketball_bets
FROM short_long_wagers
GROUP BY 1
ORDER BY 1
            
""").df()

,retention_bucket,pct_nhl_bets,pct_nba_bets,pct_mlb_bets,pct_champions_league_bets,pct_nfl_bets,pct_college_football_bets,pct_college_basketball_bets
0,1-3 months,5.23,35.02,7.69,0.71,33.00,5.06,17.41
1,6+ months,5.56,31.04,13.97,0.48,31.90,7.78,13.53


**Note:** The above percentages will sum to >100%, but that makes sense because a parlay can bet on multiple sports.

**Part 1d: Where do we see the most 1 month players?**

In [9]:
duckdb.sql ("""
WITH sports_encoded AS (
        SELECT *
        , CASE WHEN LOWER (sport) == 'nhl' THEN 1 ELSE 0 END AS is_nhl_bet
        , CASE WHEN LOWER (sport) == 'nba' THEN 1 ELSE 0 END AS is_nba_bet
        , CASE WHEN LOWER (sport) == 'mlb' THEN 1 ELSE 0 END AS is_mlb_bet
        , CASE WHEN LOWER (sport) == 'champions league' THEN 1 ELSE 0 END AS is_champions_league_bet
        , CASE WHEN LOWER (sport) == 'nfl' THEN 1 ELSE 0 END AS is_nfl_bet
        , CASE WHEN LOWER (sport) == 'college football' THEN 1 ELSE 0 END AS is_college_football_bet
        , CASE WHEN LOWER (sport) == 'college basketball' THEN 1 ELSE 0 END AS is_college_basketball_bet  
        FROM bets
        )
, wager_level AS (
        SELECT player_id
        , wager_id
        , bet_type
        , handle
        , ggr
        , MIN(CAST (placed_date AS TIMESTAMP)) AS placed_date
        , MAX(CAST (settled_date AS TIMESTAMP)) AS settled_date
        , MAX(CAST (event_start AS TIMESTAMP)) AS event_start
        , MAX(is_nhl_bet) AS is_nhl_bet
        , MAX(is_nba_bet) AS is_nba_bet
        , MAX(is_mlb_bet) AS is_mlb_bet
        , MAX(is_champions_league_bet) AS is_champions_league_bet
        , MAX(is_nfl_bet) AS is_nfl_bet
        , MAX(is_college_football_bet) AS is_college_football_bet
        , MAX(is_college_basketball_bet) AS is_college_basketball_bet
        FROM sports_encoded
        GROUP BY 1, 2, 3, 4, 5          
        )
, player_months AS (
        SELECT player_id
        , COUNT(DISTINCT date_trunc('month', placed_date)) AS active_months
        FROM wager_level
        GROUP BY 1
        )
, player_buckets AS (
        SELECT player_id
        , active_months
        , CASE
                WHEN active_months == 1 THEN '1 month'
                WHEN active_months == 2 THEN '2 month'
                WHEN active_months == 3 THEN '3 month'
                WHEN active_months == 4 THEN '4 month'
                WHEN active_months == 5 THEN '5 month'
                ELSE '6+ months'
        END AS retention_bucket
        FROM player_months
            )
, short_long AS (
        SELECT player_id
        , active_months
        , retention_bucket
        FROM player_buckets
        WHERE retention_bucket IN ('1 month', '6+ months')
        )
, short_long_wagers AS (
        SELECT sl.player_id
        , sl.active_months
        , sl.retention_bucket
        , w.wager_id
        , w.placed_date
        , w.bet_type
        , w.handle
        , w.ggr
        , is_nhl_bet
        , is_nba_bet
        , is_mlb_bet
        , is_champions_league_bet
        , is_nfl_bet
        , is_college_football_bet
        , is_college_basketball_bet
        FROM short_long sl
        LEFT JOIN wager_level w ON sl.player_id = w.player_id
        )
, one_month_first AS (
        SELECT player_id
        , date_trunc ('month', MIN (placed_date)) AS first_month
        FROM short_long_wagers
        WHERE retention_bucket = '1 month'
        GROUP BY 1
        )
            
SELECT first_month
, COUNT(*) AS num_players
FROM one_month_first
GROUP BY 1
ORDER BY 1
            
""").df()

,first_month,num_players
0,2021-04-01,543
1,2021-05-01,275
2,2021-06-01,286
3,2021-07-01,323
4,2021-08-01,173
5,2021-09-01,1499
6,2021-10-01,1338
7,2021-11-01,1007
8,2021-12-01,821
9,2022-01-01,1490


**Part 1e: For patrons that only bet in February 2022, what sport was it?**

In [10]:
duckdb.sql ("""
WITH sports_encoded AS (
        SELECT *
        , CASE WHEN LOWER (sport) == 'nhl' THEN 1 ELSE 0 END AS is_nhl_bet
        , CASE WHEN LOWER (sport) == 'nba' THEN 1 ELSE 0 END AS is_nba_bet
        , CASE WHEN LOWER (sport) == 'mlb' THEN 1 ELSE 0 END AS is_mlb_bet
        , CASE WHEN LOWER (sport) == 'champions league' THEN 1 ELSE 0 END AS is_champions_league_bet
        , CASE WHEN LOWER (sport) == 'nfl' THEN 1 ELSE 0 END AS is_nfl_bet
        , CASE WHEN LOWER (sport) == 'college football' THEN 1 ELSE 0 END AS is_college_football_bet
        , CASE WHEN LOWER (sport) == 'college basketball' THEN 1 ELSE 0 END AS is_college_basketball_bet  
        FROM bets
        )
, wager_level AS (
        SELECT player_id
        , wager_id
        , bet_type
        , handle
        , ggr
        , MIN(CAST (placed_date AS TIMESTAMP)) AS placed_date
        , MAX(CAST (settled_date AS TIMESTAMP)) AS settled_date
        , MAX(CAST (event_start AS TIMESTAMP)) AS event_start
        , MAX(is_nhl_bet) AS is_nhl_bet
        , MAX(is_nba_bet) AS is_nba_bet
        , MAX(is_mlb_bet) AS is_mlb_bet
        , MAX(is_champions_league_bet) AS is_champions_league_bet
        , MAX(is_nfl_bet) AS is_nfl_bet
        , MAX(is_college_football_bet) AS is_college_football_bet
        , MAX(is_college_basketball_bet) AS is_college_basketball_bet
        FROM sports_encoded
        GROUP BY 1, 2, 3, 4, 5          
        )
, player_months AS (
        SELECT player_id
        , COUNT (DISTINCT date_trunc ('month', placed_date)) AS active_months
        FROM wager_level
        GROUP BY 1
        )
, player_buckets AS (
        SELECT player_id
        , active_months
        , CASE
                WHEN active_months == 1 THEN '1 month'
                WHEN active_months == 2 THEN '2 month'
                WHEN active_months == 3 THEN '3 month'
                WHEN active_months == 4 THEN '4 month'
                WHEN active_months == 5 THEN '5 month'
                ELSE '6+ months'
        END AS retention_bucket
        FROM player_months
            )
, short_long AS (
        SELECT player_id
        , active_months
        , retention_bucket
        FROM player_buckets
        WHERE retention_bucket IN ('1 month', '6+ months')
        )
, short_long_wagers AS (
        SELECT sl.player_id
        , sl.active_months
        , sl.retention_bucket
        , w.wager_id
        , w.placed_date
        , w.bet_type
        , w.handle
        , w.ggr
        , is_nhl_bet
        , is_nba_bet
        , is_mlb_bet
        , is_champions_league_bet
        , is_nfl_bet
        , is_college_football_bet
        , is_college_basketball_bet
        FROM short_long sl
        LEFT JOIN wager_level w ON sl.player_id = w.player_id
        )
, one_month_first AS (
        SELECT player_id
        , date_trunc ('month', MIN (placed_date)) AS first_month
        FROM short_long_wagers
        WHERE retention_bucket = '1 month'
        GROUP BY 1
        )
, feb_players AS (
        SELECT player_id   
        FROM one_month_first
        WHERE first_month = '2022-02-01'  
        )
, feb_players_wagers AS (
        SELECT slw.*
        FROM short_long_wagers slw
        JOIN feb_players f on slw.player_id = f.player_id
        )

SELECT COUNT(*) AS total_wagers,
    ROUND(SUM(is_nhl_bet) / COUNT(*), 4) * 100 AS pct_nhl,
    ROUND(SUM(is_nba_bet) / COUNT(*), 4) * 100 AS pct_nba,
    ROUND(SUM(is_mlb_bet) / COUNT(*), 4) * 100 AS pct_mlb,
    ROUND(SUM(is_champions_league_bet) / COUNT(*), 4) * 100 AS pct_champions_league,
    ROUND(SUM(is_nfl_bet) / COUNT(*), 4) * 100 AS pct_nfl,
    ROUND(SUM(is_college_football_bet) / COUNT(*), 4) * 100 AS pct_college_football,
    ROUND(SUM(is_college_basketball_bet) / COUNT(*), 4) * 100 AS pct_college_basketball
FROM feb_players_wagers
            
""").df()

,total_wagers,pct_nhl,pct_nba,pct_mlb,pct_champions_league,pct_nfl,pct_college_football,pct_college_basketball
0,11206,3.64,26.97,0.00,1.37,57.55,0.00,11.81


**Part 1f: How many 2022 Superbowl bettors returned in March?**

In [11]:
duckdb.sql ("""
WITH sports_encoded AS (
        SELECT *
        , CASE WHEN LOWER (sport) == 'nhl' THEN 1 ELSE 0 END AS is_nhl_bet
        , CASE WHEN LOWER (sport) == 'nba' THEN 1 ELSE 0 END AS is_nba_bet
        , CASE WHEN LOWER (sport) == 'mlb' THEN 1 ELSE 0 END AS is_mlb_bet
        , CASE WHEN LOWER (sport) == 'champions league' THEN 1 ELSE 0 END AS is_champions_league_bet
        , CASE WHEN LOWER (sport) == 'nfl' THEN 1 ELSE 0 END AS is_nfl_bet
        , CASE WHEN LOWER (sport) == 'college football' THEN 1 ELSE 0 END AS is_college_football_bet
        , CASE WHEN LOWER (sport) == 'college basketball' THEN 1 ELSE 0 END AS is_college_basketball_bet  
        FROM bets
        )
, wager_level AS (
        SELECT player_id
        , wager_id
        , bet_type
        , handle
        , ggr
        , MIN(CAST (placed_date AS TIMESTAMP)) AS placed_date
        , MAX(CAST (settled_date AS TIMESTAMP)) AS settled_date
        , MAX(CAST (event_start AS TIMESTAMP)) AS event_start
        , MAX(is_nhl_bet) AS is_nhl_bet
        , MAX(is_nba_bet) AS is_nba_bet
        , MAX(is_mlb_bet) AS is_mlb_bet
        , MAX(is_champions_league_bet) AS is_champions_league_bet
        , MAX(is_nfl_bet) AS is_nfl_bet
        , MAX(is_college_football_bet) AS is_college_football_bet
        , MAX(is_college_basketball_bet) AS is_college_basketball_bet
        FROM sports_encoded
        GROUP BY 1, 2, 3, 4, 5          
        )
, feb_nfl_players AS (
        SELECT DISTINCT player_id
        FROM wager_level
        WHERE is_nfl_bet = 1
        AND placed_date >= TIMESTAMP '2022-02-01'
        AND placed_date <  TIMESTAMP '2022-03-01'
        )
, march_returns AS (
        SELECT DISTINCT wl.player_id
        FROM wager_level wl
        JOIN feb_nfl_players f ON wl.player_id = f.player_id
        WHERE wl.placed_date >= TIMESTAMP '2022-03-01'
)
, feb_count AS (
        SELECT COUNT (*) AS feb_nfl_players
        FROM feb_nfl_players
        )
, march_count AS (
        SELECT COUNT (*) AS feb_nfl_returned
        FROM march_returns
        )

SELECT f.feb_nfl_players
, m.feb_nfl_returned
, ROUND (m.feb_nfl_returned/f.feb_nfl_players, 4) * 100 AS 'pct_returned'
FROM feb_count f
CROSS JOIN march_count m
            
""").df()

,feb_nfl_players,feb_nfl_returned,pct_returned
0,11769,6353,53.98


**Part 1g: For 1 month bettors, what % only placed 1 bet?** 

In [12]:
duckdb.sql ("""
WITH sports_encoded AS (
        SELECT *
        , CASE WHEN LOWER (sport) == 'nhl' THEN 1 ELSE 0 END AS is_nhl_bet
        , CASE WHEN LOWER (sport) == 'nba' THEN 1 ELSE 0 END AS is_nba_bet
        , CASE WHEN LOWER (sport) == 'mlb' THEN 1 ELSE 0 END AS is_mlb_bet
        , CASE WHEN LOWER (sport) == 'champions league' THEN 1 ELSE 0 END AS is_champions_league_bet
        , CASE WHEN LOWER (sport) == 'nfl' THEN 1 ELSE 0 END AS is_nfl_bet
        , CASE WHEN LOWER (sport) == 'college football' THEN 1 ELSE 0 END AS is_college_football_bet
        , CASE WHEN LOWER (sport) == 'college basketball' THEN 1 ELSE 0 END AS is_college_basketball_bet  
        FROM bets
        )
, wager_level AS (
        SELECT player_id
        , wager_id
        , bet_type
        , handle
        , ggr
        , MIN(CAST (placed_date AS TIMESTAMP)) AS placed_date
        , MAX(CAST (settled_date AS TIMESTAMP)) AS settled_date
        , MAX(CAST (event_start AS TIMESTAMP)) AS event_start
        , MAX(is_nhl_bet) AS is_nhl_bet
        , MAX(is_nba_bet) AS is_nba_bet
        , MAX(is_mlb_bet) AS is_mlb_bet
        , MAX(is_champions_league_bet) AS is_champions_league_bet
        , MAX(is_nfl_bet) AS is_nfl_bet
        , MAX(is_college_football_bet) AS is_college_football_bet
        , MAX(is_college_basketball_bet) AS is_college_basketball_bet
        FROM sports_encoded
        GROUP BY 1, 2, 3, 4, 5          
        )
, player_months AS (
        SELECT player_id
        , COUNT (DISTINCT date_trunc ('month', placed_date)) AS active_months
        FROM wager_level
        GROUP BY 1
        )
, player_buckets AS (
        SELECT player_id
        , active_months
        , CASE
                WHEN active_months == 1 THEN '1 month'
                WHEN active_months == 2 THEN '2 month'
                WHEN active_months == 3 THEN '3 month'
                WHEN active_months == 4 THEN '4 month'
                WHEN active_months == 5 THEN '5 month'
                ELSE '6+ months'
        END AS retention_bucket
        FROM player_months
            )
, short_long AS (
        SELECT player_id
        , active_months
        , retention_bucket
        FROM player_buckets
        WHERE retention_bucket IN ('1 month', '6+ months')
        )
, short_long_wagers AS (
        SELECT sl.player_id
        , sl.active_months
        , sl.retention_bucket
        , w.wager_id
        , w.placed_date
        , w.bet_type
        , w.handle
        , w.ggr
        , is_nhl_bet
        , is_nba_bet
        , is_mlb_bet
        , is_champions_league_bet
        , is_nfl_bet
        , is_college_football_bet
        , is_college_basketball_bet
        FROM short_long sl
        LEFT JOIN wager_level w ON sl.player_id = w.player_id
        )
, one_month_wager_counts AS (
        SELECT player_id
        , COUNT(*) AS num_wagers
        FROM short_long_wagers
        WHERE retention_bucket = '1 month' 
        GROUP BY 1
        )

SELECT ROUND (SUM (CASE WHEN num_wagers = 1 THEN 1 ELSE 0 END) / COUNT(*), 4) * 100 AS pct_one_bet
FROM one_month_wager_counts
            
""").df()

,pct_one_bet
0,38.02


## Part 2: SBK's Best/Worst Month

**2a: What are the best/worst months for sportsbook?**

In [13]:
duckdb.sql ("""
WITH sports_encoded AS (
        SELECT *
        , CASE WHEN LOWER (sport) == 'nhl' THEN 1 ELSE 0 END AS is_nhl_bet
        , CASE WHEN LOWER (sport) == 'nba' THEN 1 ELSE 0 END AS is_nba_bet
        , CASE WHEN LOWER (sport) == 'mlb' THEN 1 ELSE 0 END AS is_mlb_bet
        , CASE WHEN LOWER (sport) == 'champions league' THEN 1 ELSE 0 END AS is_champions_league_bet
        , CASE WHEN LOWER (sport) == 'nfl' THEN 1 ELSE 0 END AS is_nfl_bet
        , CASE WHEN LOWER (sport) == 'college football' THEN 1 ELSE 0 END AS is_college_football_bet
        , CASE WHEN LOWER (sport) == 'college basketball' THEN 1 ELSE 0 END AS is_college_basketball_bet  
        FROM bets
        )
, wager_level AS (
        SELECT player_id
        , wager_id
        , bet_type
        , handle
        , ggr
        , MIN(CAST (placed_date AS TIMESTAMP)) AS placed_date
        , MAX(CAST (settled_date AS TIMESTAMP)) AS settled_date
        , MAX(CAST (event_start AS TIMESTAMP)) AS event_start
        , MAX(is_nhl_bet) AS is_nhl_bet
        , MAX(is_nba_bet) AS is_nba_bet
        , MAX(is_mlb_bet) AS is_mlb_bet
        , MAX(is_champions_league_bet) AS is_champions_league_bet
        , MAX(is_nfl_bet) AS is_nfl_bet
        , MAX(is_college_football_bet) AS is_college_football_bet
        , MAX(is_college_basketball_bet) AS is_college_basketball_bet
        FROM sports_encoded
        GROUP BY 1, 2, 3, 4, 5          
        )
, month_stats AS (
        SELECT date_trunc('month', placed_date) AS year_month
        , COUNT(*) AS total_bets
        , SUM(handle) AS total_handle
        , SUM(ggr) AS total_ggr
        , COUNT(DISTINCT player_id) AS active_players
        , SUM(ggr) / COUNT(DISTINCT player_Id) AS avg_ggr_per_player
        , COUNT(*) / COUNT(DISTINCT player_id) AS avg_bets_per_player
        , SUM(handle) / COUNT (*) AS avg_handle_per_bet
        FROM wager_level
        GROUP BY 1
        )
SELECT *
FROM month_stats
ORDER BY 1

""").df()

,year_month,total_bets,total_handle,total_ggr,active_players,avg_ggr_per_player,avg_bets_per_player,avg_handle_per_bet
0,2021-04-01,106993,"3,113,300.68","140,286.97",6570,21.35,16.29,29.10
1,2021-05-01,94813,"2,925,278.42","251,957.09",6034,41.76,15.71,30.85
2,2021-06-01,80118,"2,471,149.14","252,791.67",5524,45.76,14.50,30.84
3,2021-07-01,61977,"2,103,838.40","287,321.96",6376,45.06,9.72,33.95
4,2021-08-01,67893,"2,220,081.46","102,002.61",5371,18.99,12.64,32.70
5,2021-09-01,217964,"7,255,798.48","567,331.81",15025,37.76,14.51,33.29
6,2021-10-01,312663,"10,420,573.38","664,362.68",16802,39.54,18.61,33.33
7,2021-11-01,305059,"10,148,614.38","1,035,665.94",15838,65.39,19.26,33.27
8,2021-12-01,303449,"9,835,263.71","535,168.04",14908,35.90,20.35,32.41
9,2022-01-01,341254,"11,010,055.03","768,258.16",16680,46.06,20.46,32.26


**2b: What sports are being bet on from April 2021 -> March 2022?**

In [14]:
duckdb.sql ("""
WITH sports_encoded AS (
        SELECT *
        , CASE WHEN LOWER (sport) == 'nhl' THEN 1 ELSE 0 END AS is_nhl_bet
        , CASE WHEN LOWER (sport) == 'nba' THEN 1 ELSE 0 END AS is_nba_bet
        , CASE WHEN LOWER (sport) == 'mlb' THEN 1 ELSE 0 END AS is_mlb_bet
        , CASE WHEN LOWER (sport) == 'champions league' THEN 1 ELSE 0 END AS is_champions_league_bet
        , CASE WHEN LOWER (sport) == 'nfl' THEN 1 ELSE 0 END AS is_nfl_bet
        , CASE WHEN LOWER (sport) == 'college football' THEN 1 ELSE 0 END AS is_college_football_bet
        , CASE WHEN LOWER (sport) == 'college basketball' THEN 1 ELSE 0 END AS is_college_basketball_bet  
        FROM bets
        )
, wager_level AS (
        SELECT player_id
        , wager_id
        , bet_type
        , handle
        , ggr
        , MIN(CAST (placed_date AS TIMESTAMP)) AS placed_date
        , MAX(CAST (settled_date AS TIMESTAMP)) AS settled_date
        , MAX(CAST (event_start AS TIMESTAMP)) AS event_start
        , MAX(is_nhl_bet) AS is_nhl_bet
        , MAX(is_nba_bet) AS is_nba_bet
        , MAX(is_mlb_bet) AS is_mlb_bet
        , MAX(is_champions_league_bet) AS is_champions_league_bet
        , MAX(is_nfl_bet) AS is_nfl_bet
        , MAX(is_college_football_bet) AS is_college_football_bet
        , MAX(is_college_basketball_bet) AS is_college_basketball_bet
        FROM sports_encoded
        GROUP BY 1, 2, 3, 4, 5          
        )
SELECT
    date_trunc('month', placed_date) AS year_month,
    COUNT(*) AS total_wagers,
    ROUND(SUM(is_nhl_bet) / COUNT(*), 4) * 100 AS pct_nhl_bets,
    ROUND(SUM(is_nba_bet) / COUNT(*), 4) * 100 AS pct_nba_bets,
    ROUND(SUM(is_mlb_bet) / COUNT(*), 4) * 100 AS pct_mlb_bets,
    ROUND(SUM(is_champions_league_bet) / COUNT(*), 4) * 100 AS pct_champions_league_bets,
    ROUND(SUM(is_nfl_bet) / COUNT(*), 4) * 100 AS pct_nfl_bets,
    ROUND(SUM(is_college_football_bet) / COUNT(*), 4) * 100 AS pct_college_football_bets,
    ROUND(SUM(is_college_basketball_bet) / COUNT(*), 4) * 100 AS pct_college_basketball_bets
FROM wager_level
GROUP BY 1
ORDER BY 1
            
""").df()

,year_month,total_wagers,pct_nhl_bets,pct_nba_bets,pct_mlb_bets,pct_champions_league_bets,pct_nfl_bets,pct_college_football_bets,pct_college_basketball_bets
0,2021-04-01,106993,8.58,51.45,35.53,1.55,0.13,0.13,8.53
1,2021-05-01,94813,9.28,60.68,34.47,1.29,0.35,0.09,0.00
2,2021-06-01,80118,7.86,62.78,35.20,0.00,0.30,0.00,0.00
3,2021-07-01,61977,2.07,29.10,69.81,0.00,1.27,0.02,0.00
4,2021-08-01,67893,0.01,0.05,93.20,0.02,3.32,3.73,0.01
5,2021-09-01,217964,0.01,0.01,22.73,1.22,56.85,21.48,0.00
6,2021-10-01,312663,4.49,14.81,12.86,0.32,54.02,17.83,0.00
7,2021-11-01,305059,5.98,29.86,0.49,0.54,43.46,13.69,11.15
8,2021-12-01,303449,5.50,33.91,0.00,0.29,47.42,8.01,10.80
9,2022-01-01,341254,6.42,33.87,0.00,0.00,47.14,1.29,15.50


**2c: What is the split for straights vs. parlays over April 2021 -> March 2022?**

In [15]:
duckdb.sql ("""
WITH sports_encoded AS (
        SELECT *
        , CASE WHEN LOWER (sport) == 'nhl' THEN 1 ELSE 0 END AS is_nhl_bet
        , CASE WHEN LOWER (sport) == 'nba' THEN 1 ELSE 0 END AS is_nba_bet
        , CASE WHEN LOWER (sport) == 'mlb' THEN 1 ELSE 0 END AS is_mlb_bet
        , CASE WHEN LOWER (sport) == 'champions league' THEN 1 ELSE 0 END AS is_champions_league_bet
        , CASE WHEN LOWER (sport) == 'nfl' THEN 1 ELSE 0 END AS is_nfl_bet
        , CASE WHEN LOWER (sport) == 'college football' THEN 1 ELSE 0 END AS is_college_football_bet
        , CASE WHEN LOWER (sport) == 'college basketball' THEN 1 ELSE 0 END AS is_college_basketball_bet  
        FROM bets
        )
, wager_level AS (
        SELECT player_id
        , wager_id
        , bet_type
        , handle
        , ggr
        , MIN(CAST (placed_date AS TIMESTAMP)) AS placed_date
        , MAX(CAST (settled_date AS TIMESTAMP)) AS settled_date
        , MAX(CAST (event_start AS TIMESTAMP)) AS event_start
        , MAX(is_nhl_bet) AS is_nhl_bet
        , MAX(is_nba_bet) AS is_nba_bet
        , MAX(is_mlb_bet) AS is_mlb_bet
        , MAX(is_champions_league_bet) AS is_champions_league_bet
        , MAX(is_nfl_bet) AS is_nfl_bet
        , MAX(is_college_football_bet) AS is_college_football_bet
        , MAX(is_college_basketball_bet) AS is_college_basketball_bet
        FROM sports_encoded
        GROUP BY 1, 2, 3, 4, 5          
        )
SELECT
    date_trunc('month', placed_date) AS year_month
    , COUNT(*) AS total_wagers
    , ROUND(SUM(CASE WHEN LOWER(bet_type) = 'straight' THEN 1 ELSE 0 END) / COUNT(*), 4) * 100 AS pct_straight_bets
    , ROUND(SUM(CASE WHEN LOWER(bet_type) = 'parlay'  THEN 1 ELSE 0 END) / COUNT(*), 4) * 100 AS pct_parlay_bets
FROM wager_level
GROUP BY 1
ORDER BY 1
            
""").df()

,year_month,total_wagers,pct_straight_bets,pct_parlay_bets
0,2021-04-01,106993,74.35,25.65
1,2021-05-01,94813,73.34,26.66
2,2021-06-01,80118,76.60,23.40
3,2021-07-01,61977,69.93,30.07
4,2021-08-01,67893,65.25,34.75
5,2021-09-01,217964,70.87,29.13
6,2021-10-01,312663,68.34,31.66
7,2021-11-01,305059,70.44,29.56
8,2021-12-01,303449,69.37,30.63
9,2022-01-01,341254,72.53,27.47


## Part 3: Parlays vs. Straights

**Note:** For this section I decided to exclude March 2021 -> June 2021 because of records missing decimal odds in the dataset (see Quality Check notebook).

**The below queries cover the busiest period for the Sportsbook (Sept 2021 -> March 2022).**

**Part 3a:** How do parlay and straight bets differ?

In [16]:

duckdb.sql ("""
WITH sports_encoded AS (
        SELECT *
        , CASE WHEN LOWER (sport) == 'nhl' THEN 1 ELSE 0 END AS is_nhl_bet
        , CASE WHEN LOWER (sport) == 'nba' THEN 1 ELSE 0 END AS is_nba_bet
        , CASE WHEN LOWER (sport) == 'mlb' THEN 1 ELSE 0 END AS is_mlb_bet
        , CASE WHEN LOWER (sport) == 'champions league' THEN 1 ELSE 0 END AS is_champions_league_bet
        , CASE WHEN LOWER (sport) == 'nfl' THEN 1 ELSE 0 END AS is_nfl_bet
        , CASE WHEN LOWER (sport) == 'college football' THEN 1 ELSE 0 END AS is_college_football_bet
        , CASE WHEN LOWER (sport) == 'college basketball' THEN 1 ELSE 0 END AS is_college_basketball_bet  
        FROM bets
        )
, wager_level AS (
        SELECT player_id
        , wager_id
        , bet_type
        , handle
        , ggr
        , bet_result
        , COUNT (*) AS legs
        , PRODUCT (decimal_odds) AS decimal_odds
        , MIN(CAST(placed_date AS TIMESTAMP)) AS placed_date
        , MAX(CAST(settled_date AS TIMESTAMP)) AS settled_date
        , MAX(is_nhl_bet) AS is_nhl_bet
        , MAX(is_nba_bet) AS is_nba_bet
        , MAX(is_mlb_bet) AS is_mlb_bet
        , MAX(is_champions_league_bet) AS is_champions_league_bet
        , MAX(is_nfl_bet) AS is_nfl_bet
        , MAX(is_college_football_bet) AS is_college_football_bet
        , MAX(is_college_basketball_bet) AS is_college_basketball_bet
        FROM sports_encoded
        GROUP BY 1, 2, 3, 4, 5, 6
)
            
SELECT bet_type
, COUNT(*) AS total_wagers
, AVG(handle) AS avg_handle
, AVG (legs) AS avg_legs
, AVG (decimal_odds) AS avg_decimal_odds
, ROUND (COUNT (*) / SUM(COUNT (*)) OVER (), 2) * 100 as pct_of_wagers
FROM wager_level
WHERE placed_date BETWEEN '2021-09-01' AND '2022-03-31'
GROUP BY 1

""").df()

,bet_type,total_wagers,avg_handle,avg_legs,avg_decimal_odds,pct_of_wagers
0,parlay,569657,14.47,3.61,49.94,29.00
1,straight,1405036,39.78,1.00,7.29,71.00


**Part 3b:** How do patrons that only place parlays differ from other bettors?

In [17]:
duckdb.sql ("""
WITH sports_encoded AS (
        SELECT *
        , CASE WHEN LOWER (sport) == 'nhl' THEN 1 ELSE 0 END AS is_nhl_bet
        , CASE WHEN LOWER (sport) == 'nba' THEN 1 ELSE 0 END AS is_nba_bet
        , CASE WHEN LOWER (sport) == 'mlb' THEN 1 ELSE 0 END AS is_mlb_bet
        , CASE WHEN LOWER (sport) == 'champions league' THEN 1 ELSE 0 END AS is_champions_league_bet
        , CASE WHEN LOWER (sport) == 'nfl' THEN 1 ELSE 0 END AS is_nfl_bet
        , CASE WHEN LOWER (sport) == 'college football' THEN 1 ELSE 0 END AS is_college_football_bet
        , CASE WHEN LOWER (sport) == 'college basketball' THEN 1 ELSE 0 END AS is_college_basketball_bet  
        FROM bets
        )
, wager_level AS (
        SELECT player_id
        , wager_id
        , bet_type
        , handle
        , ggr
        , bet_result
        , COUNT (*) AS legs
        , PRODUCT (decimal_odds) AS decimal_odds
        , MIN(CAST(placed_date AS TIMESTAMP)) AS placed_date
        , MAX(CAST(settled_date AS TIMESTAMP)) AS settled_date
        , MAX(is_nhl_bet) AS is_nhl_bet
        , MAX(is_nba_bet) AS is_nba_bet
        , MAX(is_mlb_bet) AS is_mlb_bet
        , MAX(is_champions_league_bet) AS is_champions_league_bet
        , MAX(is_nfl_bet) AS is_nfl_bet
        , MAX(is_college_football_bet) AS is_college_football_bet
        , MAX(is_college_basketball_bet) AS is_college_basketball_bet
        FROM sports_encoded
        GROUP BY 1, 2, 3, 4, 5, 6
)
, player_stats AS (
        SELECT player_id
        , SUM(CASE WHEN LOWER(bet_type) = 'straight' THEN 1 ELSE 0 END) AS straight_bets
        , SUM(CASE WHEN LOWER(bet_type) = 'parlay'   THEN 1 ELSE 0 END) AS parlay_bets
        , COUNT (*) AS total_bets
        , SUM (handle) as total_handle
        , SUM (ggr) AS total_ggr
        , AVG(CASE WHEN LOWER(bet_type) = 'parlay' THEN legs END)  AS avg_parlay_legs
        , AVG (handle) AS avg_handle
        , AVG (decimal_odds) AS avg_odds
        FROM wager_level
        WHERE placed_date BETWEEN '2021-09-01' AND '2022-03-31'
        GROUP BY 1
        )
, cohort_players AS (
        SELECT player_id
        , total_bets
        , total_handle
        , total_ggr
        , avg_parlay_legs
        , avg_odds
        , avg_handle
        , straight_bets
        , parlay_bets
        , CASE 
            WHEN parlay_bets  > 1 AND straight_bets = 0 THEN 'parlay_only'
            WHEN straight_bets > 1 AND parlay_bets  = 0 THEN 'straight_only'
            WHEN parlay_bets  = 1 AND straight_bets = 0 THEN 'single_parlay_only'
            WHEN straight_bets = 1 AND parlay_bets  = 0 THEN 'single_straight_only'
            ELSE 'mixed'
          END AS cohort
    FROM player_stats
)

SELECT cohort
    , COUNT(*) AS patrons
    , SUM(total_bets) AS total_bets_placed
    , AVG(total_bets) AS avg_bets_per_patron
    , SUM(total_handle) AS total_handle
    , SUM (total_ggr) AS total_ggr
    , AVG(avg_handle) AS avg_handle_per_patron
    , AVG (avg_odds) AS avg_odds_per_per_patron
    , AVG(avg_parlay_legs) AS avg_parlay_legs_per_patron
FROM cohort_players
GROUP BY 1


""").df()

,cohort,patrons,total_bets_placed,avg_bets_per_patron,total_handle,total_ggr,avg_handle_per_patron,avg_odds_per_per_patron,avg_parlay_legs_per_patron
0,parlay_only,2402,"23,718.00",9.87,"312,458.75","91,364.51",14.67,96.41,4.82
1,mixed,21226,"1,851,894.00",87.25,"59,500,355.11","3,990,158.52",32.00,35.60,3.46
2,single_parlay_only,1532,"1,532.00",1.00,"23,933.54","-5,690.27",15.62,103.40,4.23
3,straight_only,5998,"94,788.00",15.80,"4,164,073.25","388,365.35",48.65,15.19,NaN
4,single_straight_only,2761,"2,761.00",1.00,"136,137.33",-698.54,49.31,16.57,NaN


**Note:** In the above query I decided to exlude patrons that bet once from parlay-only and straight_only groups to paint a better picture.

## Bonus Metrics for Slides

In [18]:
duckdb.sql ("""

SELECT COUNT (DISTINCT wager_id) AS 'bets'
, COUNT (DISTINCT player_id) AS 'players'
FROM bets



""").df()

,bets,players
0,2386487,36284
